# 04 BM25  
# Nombre: Wellington Barros

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [ ]:
import re
import math
import nltk
import numpy as np
import pandas as pd

from collections import Counter # importar Counter para contar la frecuencia de términos
from nltk.corpus import gutenberg

nltk.download("gutenberg")

def preprocesar(texto): # función para preprocesar el texto, convirtiéndolo a minúsculas y extrayendo solo palabras alfabéticas
    return re.findall(r"\b[a-z]+\b", texto.lower())

documentos = []

for fileid in gutenberg.fileids():
    texto = gutenberg.raw(fileid)
    tokens = preprocesar(texto)

    documentos.append({
        "doc_id": str(fileid),
        "titulo": str(fileid).replace(".txt", ""),
        "texto": texto,
        "tokens": tokens,
        "longitud": len(tokens)
    })

df_corpus = pd.DataFrame(documentos) # crear un DataFrame a partir de la lista de documentos

N = len(df_corpus)

df_diccionario = Counter()

for tokens in df_corpus["tokens"]:
    for termino in set(tokens):
        df_diccionario[termino] += 1

idf_diccionario = {
    termino: math.log(N / frecuencia) + 1
    for termino, frecuencia in df_diccionario.items()
}

vocabulario = sorted(df_diccionario.keys())

print(f"Número de documentos: {N}")
print(f"Tamaño del vocabulario: {len(vocabulario)}")
print(f"Longitud mínima: {df_corpus['longitud'].min()}")
print(f"Longitud media: {df_corpus['longitud'].mean():.2f}")
print(f"Longitud máxima: {df_corpus['longitud'].max()}")

df_corpus[["doc_id", "titulo", "tokens"]].head()

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\pc\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


Número de documentos: 18
Tamaño del vocabulario: 41481
Longitud mínima: 6934
Longitud media: 118632.94
Longitud máxima: 791842


,doc_id,titulo,tokens
0,austen-emma.txt,austen-emma,"[emma, by, jane, austen, volume, i, chapter, i..."
1,austen-persuasion.txt,austen-persuasion,"[persuasion, by, jane, austen, chapter, sir, w..."
2,austen-sense.txt,austen-sense,"[sense, and, sensibility, by, jane, austen, ch..."
3,bible-kjv.txt,bible-kjv,"[the, king, james, bible, the, old, testament,..."
4,blake-poems.txt,blake-poems,"[poems, by, william, blake, songs, of, innocen..."


### Interpretación

En esta celda se carga el corpus de Project Gutenberg y se aplica un preprocesamiento básico que convierte el texto a minúsculas y conserva únicamente tokens alfabéticos. También se calcula la longitud de cada documento, el tamaño del vocabulario, la frecuencia documental `df` y el valor `idf` de cada término. La frecuencia documental indica en cuántos documentos aparece un término, mientras que el IDF asigna mayor peso a las palabras poco frecuentes en la colección.

**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Representación dispersa TF-IDF
corpus_procesado = [
    " ".join(tokens)
    for tokens in df_corpus["tokens"]
]

vectorizer = TfidfVectorizer(
    vocabulary=vocabulario,
    token_pattern=r"(?u)\b\w+\b",
    smooth_idf=False,
    sublinear_tf=False,
    norm=None
)

matriz_tfidf = vectorizer.fit_transform(corpus_procesado) # crear la matriz TF-IDF a partir del corpus procesado

# Parámetros BM25
k1 = 1.2
b = 0.75
longitud_media = df_corpus["longitud"].mean()

def idf_bm25(df_termino): # función para calcular el IDF de un término según la fórmula BM25
    return math.log(
        1 + (N - df_termino + 0.5) / (df_termino + 0.5)
    )

def puntuacion_bm25(tokens_consulta, tokens_documento): # función para calcular la puntuación BM25 de un documento dado una consulta
    frecuencias = Counter(tokens_documento)
    longitud_documento = len(tokens_documento)
    puntuacion = 0.0

    for termino in set(tokens_consulta):
        if termino not in frecuencias:
            continue

        frecuencia = frecuencias[termino]
        df_termino = df_diccionario[termino]
        idf = idf_bm25(df_termino)

        numerador = frecuencia * (k1 + 1)
        denominador = (
            frecuencia
            + k1 * (
                1 - b
                + b * longitud_documento / longitud_media
            )
        )

        puntuacion += idf * numerador / denominador

    return puntuacion

consultas = {
    "Q1": ("whale sea captain", "Buscar documentos relacionados con ballenas, mar y capitanes."),
    "Q2": ("alice rabbit wonderland", "Buscar historias sobre Alicia, el conejo y el país de las maravillas."),
    "Q3": ("pride prejudice marriage", "Buscar documentos sobre orgullo, prejuicio y matrimonio."),
    "Q4": ("love woman man", "Buscar documentos sobre relaciones amorosas entre hombres y mujeres."),
    "Q5": ("science astronomy physics", "Buscar contenido relacionado con ciencia, astronomía y física."),
    "Q6": ("king war", "Buscar documentos relacionados con reyes y guerras."),
    "Q7": ("time", "Buscar documentos que contengan el término frecuente relacionado con tiempo."),
    "Q8": ("xylophone", "Probar una consulta cuyo término probablemente no exista en el vocabulario.")
}

resultados = {}

for codigo, (texto_consulta, objetivo) in consultas.items(): # iterar sobre cada consulta y su objetivo
    tokens_consulta = preprocesar(texto_consulta)

    # Ranking TF-IDF + coseno
    consulta_procesada = " ".join(tokens_consulta)
    vector_consulta = vectorizer.transform([consulta_procesada])

    similitudes = cosine_similarity(
        vector_consulta,
        matriz_tfidf
    )[0]

    ranking_tfidf = []

    for indice, score in enumerate(similitudes):
        if score > 0:
            ranking_tfidf.append({
                "doc_id": str(df_corpus.iloc[indice]["doc_id"]),
                "titulo": df_corpus.iloc[indice]["titulo"],
                "score": float(score)
            })

    ranking_tfidf.sort(
        key=lambda fila: (-fila["score"], fila["doc_id"])
    )

    ranking_tfidf = ranking_tfidf[:10]

    # Ranking BM25
    ranking_bm25 = []

    for indice, tokens_documento in enumerate(df_corpus["tokens"]):
        score = puntuacion_bm25(tokens_consulta, tokens_documento)

        if score > 0:
            ranking_bm25.append({
                "doc_id": str(df_corpus.iloc[indice]["doc_id"]),
                "titulo": df_corpus.iloc[indice]["titulo"],
                "score": float(score)
            })

    ranking_bm25.sort(
        key=lambda fila: (-fila["score"], fila["doc_id"])
    )

    ranking_bm25 = ranking_bm25[:10]

    resultados[codigo] = {
        "consulta": texto_consulta,
        "objetivo": objetivo,
        "tokens": tokens_consulta,
        "tfidf": ranking_tfidf,
        "bm25": ranking_bm25
    }

    print(f"\n{codigo}: {texto_consulta}")
    print(f"Objetivo: {objetivo}")
    print("\nTF-IDF + coseno:")

    for posicion, fila in enumerate(ranking_tfidf, start=1):
        print(
            f"{posicion:2}. {fila['doc_id']:20} "
            f"{fila['score']:.6f}"
        )

    print("\nBM25:")

    for posicion, fila in enumerate(ranking_bm25, start=1):
        print(
            f"{posicion:2}. {fila['doc_id']:20} "
            f"{fila['score']:.6f}"
        )


Q1: whale sea captain
Objetivo: Buscar documentos relacionados con ballenas, mar y capitanes.

TF-IDF + coseno:
 1. melville-moby_dick.txt 0.109914
 2. austen-persuasion.txt 0.038019
 3. whitman-leaves.txt   0.007754
 4. bryant-stories.txt   0.006693
 5. chesterton-brown.txt 0.006133
 6. milton-paradise.txt  0.003671
 7. chesterton-ball.txt  0.003211
 8. bible-kjv.txt        0.003022
 9. edgeworth-parents.txt 0.002984
10. chesterton-thursday.txt 0.002810

BM25:
 1. melville-moby_dick.txt 4.180994
 2. bryant-stories.txt   3.717055
 3. whitman-leaves.txt   3.624014
 4. bible-kjv.txt        2.524921
 5. shakespeare-hamlet.txt 2.150941
 6. chesterton-ball.txt  1.917719
 7. austen-persuasion.txt 1.817317
 8. chesterton-brown.txt 1.765332
 9. edgeworth-parents.txt 1.738045
10. chesterton-thursday.txt 1.414927

Q2: alice rabbit wonderland
Objetivo: Buscar historias sobre Alicia, el conejo y el país de las maravillas.

TF-IDF + coseno:
 1. carroll-alice.txt    0.236411
 2. burgess-busterbrown

### Interpretación

En esta celda se construyen los rankings utilizando dos modelos. TF-IDF representa los documentos mediante una matriz dispersa y utiliza la similitud coseno para medir la relación con cada consulta. BM25 considera la frecuencia del término, la rareza del término y la longitud del documento. El parámetro `k1=1.2` controla la saturación de la frecuencia, mientras que `b=0.75` controla la normalización por longitud. Las listas se ordenan por puntuación descendente y, en caso de empate, por identificador ascendente.

**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [ ]:
def comparar_rankings(codigo): # función para comparar los rankings TF-IDF y BM25 de una consulta dada
    resultado = resultados[codigo]

    ranking_tfidf = resultado["tfidf"]
    ranking_bm25 = resultado["bm25"]

    maximo = max(
        len(ranking_tfidf),
        len(ranking_bm25)
    )

    filas = []

    for posicion in range(maximo): # iterar sobre cada posición en los rankings
        tfidf_fila = (
            ranking_tfidf[posicion]
            if posicion < len(ranking_tfidf)
            else None
        )

        bm25_fila = (
            ranking_bm25[posicion]
            if posicion < len(ranking_bm25)
            else None
        )

        filas.append({
            "Posición": posicion + 1,
            "ID y título TF-IDF": (
                f"{tfidf_fila['doc_id']} - {tfidf_fila['titulo']}"
                if tfidf_fila else ""
            ),
            "Coseno": (
                tfidf_fila["score"]
                if tfidf_fila else np.nan
            ),
            "ID y título BM25": (
                f"{bm25_fila['doc_id']} - {bm25_fila['titulo']}"
                if bm25_fila else ""
            ),
            "Score BM25": (
                bm25_fila["score"]
                if bm25_fila else np.nan
            )
        })

    tabla = pd.DataFrame(filas)

    ids_tfidf = {
        fila["doc_id"]
        for fila in ranking_tfidf
    }

    ids_bm25 = {
        fila["doc_id"]
        for fila in ranking_bm25
    }

    ids_compartidos = ids_tfidf.intersection(ids_bm25)

    print(f"\n{'=' * 90}")
    print(f"{codigo}: {resultado['consulta']}")
    print(f"IDs compartidos en top 10: {len(ids_compartidos)}")
    print(f"Tamaño real lista TF-IDF: {len(ranking_tfidf)}")
    print(f"Tamaño real lista BM25: {len(ranking_bm25)}")

    if ranking_tfidf:
        print(
            f"Primer documento TF-IDF: "
            f"{ranking_tfidf[0]['doc_id']}"
        )
    else:
        print("Primer documento TF-IDF: lista vacía")

    if ranking_bm25:
        print(
            f"Primer documento BM25: "
            f"{ranking_bm25[0]['doc_id']}"
        )
    else:
        print("Primer documento BM25: lista vacía")

    display(tabla)

for codigo in consultas:
    comparar_rankings(codigo)


Q1: whale sea captain
IDs compartidos en top 10: 9
Tamaño real lista TF-IDF: 10
Tamaño real lista BM25: 10
Primer documento TF-IDF: melville-moby_dick.txt
Primer documento BM25: melville-moby_dick.txt


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,melville-moby_dick.txt - melville-moby_dick,0.109914,melville-moby_dick.txt - melville-moby_dick,4.180994
1,2,austen-persuasion.txt - austen-persuasion,0.038019,bryant-stories.txt - bryant-stories,3.717055
2,3,whitman-leaves.txt - whitman-leaves,0.007754,whitman-leaves.txt - whitman-leaves,3.624014
3,4,bryant-stories.txt - bryant-stories,0.006693,bible-kjv.txt - bible-kjv,2.524921
4,5,chesterton-brown.txt - chesterton-brown,0.006133,shakespeare-hamlet.txt - shakespeare-hamlet,2.150941
5,6,milton-paradise.txt - milton-paradise,0.003671,chesterton-ball.txt - chesterton-ball,1.917719
6,7,chesterton-ball.txt - chesterton-ball,0.003211,austen-persuasion.txt - austen-persuasion,1.817317
7,8,bible-kjv.txt - bible-kjv,0.003022,chesterton-brown.txt - chesterton-brown,1.765332
8,9,edgeworth-parents.txt - edgeworth-parents,0.002984,edgeworth-parents.txt - edgeworth-parents,1.738045
9,10,chesterton-thursday.txt - chesterton-thursday,0.002810,chesterton-thursday.txt - chesterton-thursday,1.414927



Q2: alice rabbit wonderland
IDs compartidos en top 10: 7
Tamaño real lista TF-IDF: 7
Tamaño real lista BM25: 7
Primer documento TF-IDF: carroll-alice.txt
Primer documento BM25: carroll-alice.txt


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,carroll-alice.txt - carroll-alice,0.236411,carroll-alice.txt - carroll-alice,9.869988
1,2,burgess-busterbrown.txt - burgess-busterbrown,0.005558,chesterton-thursday.txt - chesterton-thursday,6.040322
2,3,bryant-stories.txt - bryant-stories,0.001796,burgess-busterbrown.txt - burgess-busterbrown,2.272244
3,4,chesterton-thursday.txt - chesterton-thursday,0.000766,bryant-stories.txt - bryant-stories,2.215025
4,5,chesterton-ball.txt - chesterton-ball,0.000436,edgeworth-parents.txt - edgeworth-parents,2.070316
5,6,chesterton-brown.txt - chesterton-brown,0.000379,chesterton-ball.txt - chesterton-ball,1.915725
6,7,edgeworth-parents.txt - edgeworth-parents,0.000211,chesterton-brown.txt - chesterton-brown,1.835955



Q3: pride prejudice marriage
IDs compartidos en top 10: 9
Tamaño real lista TF-IDF: 10
Tamaño real lista BM25: 10
Primer documento TF-IDF: austen-persuasion.txt
Primer documento BM25: austen-persuasion.txt


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,austen-persuasion.txt - austen-persuasion,0.003820,austen-persuasion.txt - austen-persuasion,2.921023
1,2,austen-sense.txt - austen-sense,0.003324,austen-sense.txt - austen-sense,2.752780
2,3,austen-emma.txt - austen-emma,0.002413,austen-emma.txt - austen-emma,2.634119
3,4,shakespeare-hamlet.txt - shakespeare-hamlet,0.001721,edgeworth-parents.txt - edgeworth-parents,2.180192
4,5,blake-poems.txt - blake-poems,0.001685,chesterton-brown.txt - chesterton-brown,2.001020
5,6,milton-paradise.txt - milton-paradise,0.001574,chesterton-ball.txt - chesterton-ball,1.588084
6,7,whitman-leaves.txt - whitman-leaves,0.001230,whitman-leaves.txt - whitman-leaves,1.220684
7,8,chesterton-thursday.txt - chesterton-thursday,0.001026,chesterton-thursday.txt - chesterton-thursday,1.181256
8,9,edgeworth-parents.txt - edgeworth-parents,0.000860,shakespeare-hamlet.txt - shakespeare-hamlet,1.139279
9,10,chesterton-brown.txt - chesterton-brown,0.000633,milton-paradise.txt - milton-paradise,1.131800


... (6 outputs left out)


Q7: time
IDs compartidos en top 10: 9
Tamaño real lista TF-IDF: 10
Tamaño real lista BM25: 10
Primer documento TF-IDF: shakespeare-macbeth.txt
Primer documento BM25: edgeworth-parents.txt


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,shakespeare-macbeth.txt - shakespeare-macbeth,0.027176,edgeworth-parents.txt - edgeworth-parents,0.058375
1,2,burgess-busterbrown.txt - burgess-busterbrown,0.026077,austen-sense.txt - austen-sense,0.058373
2,3,carroll-alice.txt - carroll-alice,0.023673,austen-emma.txt - austen-emma,0.058351
3,4,austen-sense.txt - austen-sense,0.020727,melville-moby_dick.txt - melville-moby_dick,0.058328
4,5,edgeworth-parents.txt - edgeworth-parents,0.020272,austen-persuasion.txt - austen-persuasion,0.058310
5,6,austen-persuasion.txt - austen-persuasion,0.018850,bryant-stories.txt - bryant-stories,0.058277
6,7,bryant-stories.txt - bryant-stories,0.018728,carroll-alice.txt - carroll-alice,0.058254
7,8,austen-emma.txt - austen-emma,0.018211,whitman-leaves.txt - whitman-leaves,0.058244
8,9,shakespeare-hamlet.txt - shakespeare-hamlet,0.015936,burgess-busterbrown.txt - burgess-busterbrown,0.058157
9,10,melville-moby_dick.txt - melville-moby_dick,0.015505,shakespeare-macbeth.txt - shakespeare-macbeth,0.058116



Q8: xylophone
IDs compartidos en top 10: 0
Tamaño real lista TF-IDF: 0
Tamaño real lista BM25: 0
Primer documento TF-IDF: lista vacía
Primer documento BM25: lista vacía


""


### Interpretación

La comparación permite observar que TF-IDF y BM25 pueden producir rankings diferentes aunque utilicen los mismos documentos y consultas. TF-IDF se basa principalmente en los pesos de los términos y la similitud coseno, mientras que BM25 limita el efecto de repetir muchas veces un término y corrige la ventaja que pueden tener los documentos demasiado largos. La intersección de los top 10 muestra cuántos documentos son recuperados por ambos métodos, pero no necesariamente en la misma posición.

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [ ]:
def aporte_bm25(termino, tokens_documento): # sirve para calcular el aporte de un término específico a la puntuación BM25 de un documento dado
    frecuencias = Counter(tokens_documento)
    frecuencia = frecuencias.get(termino, 0)

    if frecuencia == 0 or termino not in df_diccionario: # si el término no está presente en el documento o no está en el diccionario, su aporte es 0
        return 0.0

    df_termino = df_diccionario[termino] # obtener la frecuencia de documentos que contienen el término
    idf = idf_bm25(df_termino)
    longitud_documento = len(tokens_documento)

    numerador = frecuencia * (k1 + 1) # calcular el numerador de la fórmula BM25
    denominador = (
        frecuencia
        + k1 * (
            1 - b
            + b * longitud_documento / longitud_media
        )
    )

    return idf * numerador / denominador


# Buscar automáticamente una consulta donde cambie el orden
consulta_elegida = None
doc_a = None
doc_b = None

for codigo, resultado in resultados.items():  # iterar sobre cada consulta y sus resultados
    posiciones_tfidf = {
        fila["doc_id"]: posicion + 1
        for posicion, fila in enumerate(resultado["tfidf"])
    }

    posiciones_bm25 = { # crear un diccionario que mapea cada doc_id a su posición en el ranking BM25
        fila["doc_id"]: posicion + 1
        for posicion, fila in enumerate(resultado["bm25"])
    }

    documentos_comunes = set(posiciones_tfidf).intersection(
        posiciones_bm25
    )

    documentos_ordenados = sorted( # ordenar los documentos comunes por la diferencia absoluta de sus posiciones en los rankings TF-IDF y BM25
        documentos_comunes,
        key=lambda doc_id: abs(
            posiciones_tfidf[doc_id] - posiciones_bm25[doc_id]
        ),
        reverse=True
    )

    if documentos_ordenados:
        doc_a = documentos_ordenados[0]

        for candidato in documentos_ordenados[1:]:
            if (
                posiciones_tfidf[candidato]
                != posiciones_bm25[candidato]
            ):
                doc_b = candidato
                consulta_elegida = codigo
                break

        if consulta_elegida is not None:
            break

if consulta_elegida is not None: # si se encontró una consulta con dos documentos compartidos en posiciones diferentes, calcular los aportes de cada término a la puntuación BM25 de ambos documentos
    resultado = resultados[consulta_elegida]
    tokens_consulta = resultado["tokens"]

    indices = {
        str(doc_id): indice
        for indice, doc_id in enumerate(df_corpus["doc_id"])
    }

    filas_aportes = []

    for doc_id in [doc_a, doc_b]:
        indice = indices[doc_id]
        tokens_documento = df_corpus.iloc[indice]["tokens"]
        contador = Counter(tokens_documento)

        for termino in sorted(set(tokens_consulta)):
            filas_aportes.append({
                "Consulta": resultado["consulta"],
                "Documento": doc_id,
                "Longitud |d|": len(tokens_documento),
                "Término": termino,
                "f(t,d)": contador.get(termino, 0),
                "df(t)": df_diccionario.get(termino, 0),
                "Aporte BM25": aporte_bm25(
                    termino,
                    tokens_documento
                )
            })

    tabla_aportes = pd.DataFrame(filas_aportes)

    print(f"Consulta elegida: {consulta_elegida}")
    print(f"Documento A: {doc_a}")
    print(f"Documento B: {doc_b}")
    print(f"k1 = {k1}")
    print(f"b = {b}")

    display(tabla_aportes)
else:
    print(
        "No se encontró una consulta con dos documentos compartidos "
        "en posiciones diferentes."
    )

Consulta elegida: Q1
Documento A: austen-persuasion.txt
Documento B: bible-kjv.txt
k1 = 1.2
b = 0.75


,Consulta,Documento,Longitud |d|,Término,"f(t,d)",df(t),Aporte BM25
0,whale sea captain,austen-persuasion.txt,84121,captain,303,9,1.520217
1,whale sea captain,austen-persuasion.txt,84121,sea,21,16,0.297100
2,whale sea captain,austen-persuasion.txt,84121,whale,0,6,0.000000
3,whale sea captain,bible-kjv.txt,791842,captain,139,9,1.458732
4,whale sea captain,bible-kjv.txt,791842,sea,400,16,0.305555
5,whale sea captain,bible-kjv.txt,791842,whale,3,6,0.760633


El parámetro k1 controla la saturación de la frecuencia del término. Esto significa que cuando una palabra aparece muchas veces en un documento, cada aparición adicional aporta una menor cantidad de puntuación.

El parámetro b controla la normalización por longitud del documento. Cuando se utiliza un valor de b = 0.75, los documentos largos reciben una penalización relativa, especialmente cuando los términos aparecen de manera menos concentrada.

Por otro lado, df(t) permite determinar qué tan frecuente o raro es un término dentro de la colección de documentos. Los términos que aparecen en menos documentos tienen un mayor valor de IDF y, por lo tanto, aportan una puntuación más alta al cálculo de BM25.

Finalmente, a diferencia de TF-IDF, que puede favorecer a los documentos largos debido a que acumula las frecuencias de los términos, BM25 controla mejor este efecto mediante la normalización por longitud del documento y la saturación de la frecuencia de los términos.